In [ ]:
!pip install trl -qqq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 544.8/544.8 kB 14.2 MB/s eta 0:00:00


In [ ]:
import random
import torch
from datasets import load_dataset
from transformers import set_seed, AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, SFTConfig

import os
# Make sure nothing auto-compiles the model
os.environ.pop("TORCH_COMPILE", None)
os.environ["TORCHDYNAMO_DISABLE"] = "1"

import torch
# Optional but helpful
try:
    import torch._dynamo as dynamo
    dynamo.reset()
    dynamo.config.suppress_errors = True
except Exception:
    pass

set_seed(42)
torch.set_float32_matmul_precision("high")  # enable TF32 matmul if available

def build_chat_prompt(tokenizer, system_msg, user_msg, for_generation=False):
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=for_generation,
    )

from transformers import GenerationConfig

from transformers import GenerationConfig

def generate_translations(model, tokenizer, english_texts, max_new_tokens=64):
    model.eval()

    # Minimal config so no extra flags leak in
    gen_cfg = GenerationConfig(
        do_sample=False,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

    outs = []
    with torch.inference_mode():
        for text in english_texts:
            inputs = tokenizer.apply_chat_template(
                [
                    {"role": "system", "content": "You are a professional translator from English to Spanish. Respond with only the translation."},
                    {"role": "user", "content": text},
                ],
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt",
            ).to(model.device)

            gen_ids = model.generate(
                input_ids=inputs,
                generation_config=gen_cfg,
            )

            # Keep only newly generated tokens
            new_tokens = gen_ids[0, inputs.shape[-1]:]
            pred = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
            outs.append((text, pred))
    return outs

    model.eval()

    # Build a fresh generation config from the model config
    gen_cfg = GenerationConfig.from_model_config(model.config)
    gen_cfg.do_sample = False
    gen_cfg.max_new_tokens = max_new_tokens
    gen_cfg.eos_token_id = tokenizer.eos_token_id
    gen_cfg.pad_token_id = tokenizer.eos_token_id

    outs = []
    with torch.inference_mode():
        for text in english_texts:
            prompt = tokenizer.apply_chat_template(
                [
                    {"role": "system", "content": "You are a professional translator from English to Spanish. Respond with only the translation."},
                    {"role": "user", "content": text},
                ],
                tokenize=False,
                add_generation_prompt=True,
            )
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            gen = model.generate(
                **inputs,
                generation_config=gen_cfg,  # no top_p or top_k passed
            )
            outs.append((text, tokenizer.decode(gen[0], skip_special_tokens=True).strip()))
    return outs

def print_check(title, pairs):
    print("=" * 80)
    print(title)
    print("=" * 80)
    for src, pred in pairs:
        print(f"EN: {src}\nES: {pred}\n")

def SFT_with_checks(
    model_name: str = "google/gemma-3-270m-it",
    pair: str = "en-es",
    english_texts=None,
    lr: float = 5e-5,
    epochs: int = 2,
    bs: int = 4,              # smaller per-device batch to save memory
    gas: int = 16,
    mseqlen: int = 1024,      # shorter context helps VRAM
    output_dir: str = "./SFT-OPUS-en-es",
    train_take: int = 10000,  # small subset of train dataset
    eval_take: int = 2000,
    max_chars: int = 256,     # drop very long examples
):
    if english_texts is None:
        english_texts = [
            "Good morning, everyone",
            "I will send you the report later",
            "Artificial intelligence is transforming many industries",
            "Can you help me translate this sentence",
            "We need to schedule a meeting for next week",
        ]

    compute_dtype = torch.bfloat16

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.model_max_length = mseqlen
    tokenizer.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=compute_dtype,
        device_map="auto",
        attn_implementation="eager"
    )
    model.gradient_checkpointing_enable()  # NEW: save VRAM
    model.config.use_cache = False         # NEW: disable KV cache during training

    # pre SFT check with chat template
    pre_pairs = generate_translations(model, tokenizer, english_texts, max_new_tokens=64)
    print_check("Zero shot translations before SFT", pre_pairs)

    # load OPUS-100 and format with chat template
    src_lang, tgt_lang = pair.split("-")
    ds_all = load_dataset("Helsinki-NLP/opus-100", pair)
    train_raw = ds_all["train"]
    valid_raw = ds_all.get("validation", None)

    def row_ok(r):
        s = r["translation"][src_lang]
        t = r["translation"][tgt_lang]
        return len(s) <= max_chars and len(t) <= max_chars

    train_raw = train_raw.filter(row_ok)  # NEW
    if valid_raw is not None:
        valid_raw = valid_raw.filter(row_ok)  # NEW

    # downsample for faster iteration
    if train_take and len(train_raw) > train_take:
        train_raw = train_raw.shuffle(seed=42).select(range(train_take))  # NEW: random subset
    if valid_raw and eval_take and len(valid_raw) > eval_take:
        valid_raw = valid_raw.shuffle(seed=43).select(range(eval_take))
    else:
        # fall back to a small held out split if no validation split
        tmp = train_raw.train_test_split(test_size=0.01, seed=42)
        train_raw, valid_raw = tmp["train"], tmp["test"]

    def to_text(row):
        src = row["translation"][src_lang]
        tgt = row["translation"][tgt_lang]
        # training text includes the assistant reply
        txt = tokenizer.apply_chat_template(
            [
                {"role": "system", "content": "You are a professional translator from English to Spanish. Respond with only the translation."},
                {"role": "user", "content": src},
                {"role": "assistant", "content": tgt},
            ],
            tokenize=False,
        )
        row["text"] = txt + tokenizer.eos_token
        return row

    train_ds = train_raw.map(to_text, num_proc=1, load_from_cache_file=False)
    eval_ds = valid_raw.map(to_text, num_proc=1, load_from_cache_file=False)

    from trl import SFTTrainer, SFTConfig

    sft_config = SFTConfig(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=bs,
        gradient_accumulation_steps=gas,
        learning_rate=lr,
        lr_scheduler_type="linear",
        warmup_ratio=0.03,
        logging_steps=25,
        save_strategy="steps",
        save_steps=6000,
        bf16=True,
        optim="adamw_torch",
        dataset_text_field="text",
        packing=False,
        report_to="none",
        torch_compile=False,
    )

    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        processing_class=tokenizer,
    )

    trainer.train()
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)

    # post SFT check
    # Re-enable cache for inference
    model.config.use_cache = True  # NEW
    ft_model = AutoModelForCausalLM.from_pretrained(
        output_dir, torch_dtype=compute_dtype, device_map="auto"
    )
    post_pairs = generate_translations(ft_model, tokenizer, english_texts, max_new_tokens=64)
    print_check("Translations after SFT", post_pairs)

    return ft_model, tokenizer

if __name__ == "__main__":
    SFT_with_checks(
        model_name="google/gemma-3-270m-it",
        pair="en-es",
        english_texts=[
            "Good morning, everyone",
            "I will send you the report later",
            "Artificial intelligence is transforming many industries",
            "Can you help me translate this sentence",
            "We need to schedule a meeting for next week",
        ],
    )


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

...

Adding EOS to eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Step,Training Loss
25,2.636100
50,1.473000
75,1.408500
100,1.422900
125,1.408000
150,1.379700
175,1.284100
200,1.213500
225,1.227900
250,1.215500


Translations after SFT
EN: Good morning, everyone
ES: Buenos días a todos

EN: I will send you the report later
ES: Yo te enviasaré la nota para que se apunte a algún otro momento.

EN: Artificial intelligence is transforming many industries
ES: La inteligencia artificial está transformando al mundo.

EN: Can you help me translate this sentence
ES: ¿Me entienden?

EN: We need to schedule a meeting for next week
ES: Necesitamos schedulear una reunión en el próximo fin de semana.

